In [82]:
%%writefile practice92.cpp

// Задание 2: Распределённое решение системы линейных уравнений методом Гаусса
// 1. Процесс с "rank = 0" создаёт матрицу коэффициентов A размером NxN и вектор правых частей b.
// 2. Разделите строки матрицы между процессами с помощью функции "MPI_Scatter".
// 3. Реализуйте следующие шаги метода Гаусса:
 //- Прямой ход: каждый процесс выполняет вычитание строк для своей части матрицы.
 //- Обратный ход: соберите результаты на процессе с "rank = 0" и завершите вычисления.
// 4. Выведите решение системы уравнений на экран.
#include <mpi.h>            // Подключение библиотеки MPI для параллельного программирования
#include <iostream>         // Для ввода-вывода (cout, cerr)
#include <vector>           // Для использования динамических массивов std::vector
#include <iomanip>          // Для форматированного вывода (setw, setprecision)
#include <cstdlib>          // Для функций стандартной библиотеки, например atoi (конвертация строки в число)

int main(int argc, char* argv[]) {
    MPI_Init(&argc, &argv); // Инициализация MPI, передача аргументов командной строки в MPI

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank); // Получение ранга текущего процесса
    MPI_Comm_size(MPI_COMM_WORLD, &size); // Получение общего числа процессов

    // Получаем размер матрицы из параметра программы
    if (argc < 2) {                     // Проверка, передан ли аргумент N (размер матрицы)
        if (rank == 0) std::cerr << "Usage: " << argv[0] << " N\n"; // Вывод ошибки на rank 0
        MPI_Finalize();                  // Завершение работы MPI
        return 1;                        // Завершение программы с кодом ошибки
    }
    int N = std::atoi(argv[1]);          // Преобразуем строку в число, задаем размер матрицы N

    // Процесс 0 создаёт матрицу A и вектор b
    std::vector<double> A_full(N * N);   // Матрица NxN хранится в виде одномерного массива
    std::vector<double> b_full(N);       // Вектор правой части системы

    if (rank == 0) {                     // Только процесс 0 создаёт исходные данные
        // Пример: случайная матрица и вектор
        for (int i = 0; i < N; i++) {    // Заполнение вектора b случайными числами
            b_full[i] = rand() % 10 + 1;
            for (int j = 0; j < N; j++)  // Заполнение матрицы A случайными числами
                A_full[i * N + j] = rand() % 10 + 1;
        }

        // Вывод исходной матрицы и вектора
        std::cout << "Исходная матрица A:\n";
        for (int i = 0; i < N; i++) {
            for (int j = 0; j < N; j++)
                std::cout << std::setw(5) << A_full[i * N + j] << " "; // Вывод с шириной 5
            std::cout << "\n";
        }

        std::cout << "Исходный вектор b:\n";
        for (int i = 0; i < N; i++)
            std::cout << std::setw(5) << b_full[i] << " "; // Вывод вектора
        std::cout << "\n";
    }

    // Определяем количество строк на процесс
    int rows_per_proc = N / size;          // Минимальное число строк на процесс
    int remainder = N % size;              // Оставшиеся строки, если N не делится на size

    std::vector<int> sendcounts(size);     // Количество строк, которые получает каждый процесс
    std::vector<int> displs(size);         // Смещение начала блока строк для каждого процесса
    int offset = 0;
    for (int i = 0; i < size; i++) {       // Распределение строк между процессами
        sendcounts[i] = rows_per_proc + (i < remainder ? 1 : 0); // Дополнительная строка для первых remainder процессов
        displs[i] = offset;                // Смещение
        offset += sendcounts[i];           // Обновление смещения
    }

    int local_rows = sendcounts[rank];      // Количество строк, которое будет у данного процесса
    std::vector<double> local_A(local_rows * N); // Локальная часть матрицы
    std::vector<double> local_b(local_rows);     // Локальная часть вектора

    // Настраиваем sendcounts и displs для Scatterv (учитывая, что матрица хранится в 1D)
    std::vector<int> sendcounts_A(size), displs_A(size);
    for (int i = 0; i < size; i++) {
        sendcounts_A[i] = sendcounts[i] * N; // Количество элементов для Scatterv
        displs_A[i] = displs[i] * N;         // Смещение в массиве
    }

    // Разбрасываем матрицу и вектор
    MPI_Scatterv(A_full.data(), sendcounts_A.data(), displs_A.data(), MPI_DOUBLE,
                 local_A.data(), local_rows * N, MPI_DOUBLE, 0, MPI_COMM_WORLD); // Разброс матрицы
    MPI_Scatterv(b_full.data(), sendcounts.data(), displs.data(), MPI_DOUBLE,
                 local_b.data(), local_rows, MPI_DOUBLE, 0, MPI_COMM_WORLD);     // Разброс вектора b

    double start_time = MPI_Wtime(); // Начало измерения времени

    // Прямой ход метода Гаусса
    for (int k = 0; k < N; k++) {                    // Для каждой ведущей строки k
        std::vector<double> pivot_row(N);            // Ведущая строка
        double pivot_b;                              // Соответствующее значение вектора b

        if (rank == 0) {                             // На процессе 0 выбираем pivot
            for (int j = 0; j < N; j++)
                pivot_row[j] = A_full[k * N + j];
            pivot_b = b_full[k];
        }

        MPI_Bcast(pivot_row.data(), N, MPI_DOUBLE, 0, MPI_COMM_WORLD); // Рассылаем pivot_row всем процессам
        MPI_Bcast(&pivot_b, 1, MPI_DOUBLE, 0, MPI_COMM_WORLD);        // Рассылаем pivot_b

        for (int i_local = 0; i_local < local_rows; i_local++) {      // Обработка локальных строк
            int i_global = displs[rank] + i_local;                   // Вычисляем глобальный индекс строки
            if (i_global > k) {                                      // Только строки ниже pivot
                double factor = local_A[i_local * N + k] / pivot_row[k]; // Вычисляем коэффициент
                for (int j = k; j < N; j++)
                    local_A[i_local * N + j] -= factor * pivot_row[j];   // Вычитание строк
                local_b[i_local] -= factor * pivot_b;                   // Вычитание правой части
            }
        }

        MPI_Gatherv(local_A.data(), local_rows * N, MPI_DOUBLE,
                    A_full.data(), sendcounts_A.data(), displs_A.data(), MPI_DOUBLE,
                    0, MPI_COMM_WORLD); // Сбор обновлённой матрицы на rank 0
        MPI_Gatherv(local_b.data(), local_rows, MPI_DOUBLE,
                    b_full.data(), sendcounts.data(), displs.data(), MPI_DOUBLE,
                    0, MPI_COMM_WORLD); // Сбор обновлённого вектора b
    }

    std::vector<double> x(N, 0); // Вектор решения
    if (rank == 0) {             // Обратный ход выполняет только rank 0
        for (int i = N - 1; i >= 0; i--) {                 // Проходим строки снизу вверх
            x[i] = b_full[i];
            for (int j = i + 1; j < N; j++)
                x[i] -= A_full[i * N + j] * x[j];          // Вычитаем уже найденные элементы
            x[i] /= A_full[i * N + i];                     // Делим на диагональный элемент
        }

        double end_time = MPI_Wtime();                     // Конец измерения времени

        // Вывод решения
        std::cout << "Решение x:\n";
        for (int i = 0; i < N; i++)
            std::cout << std::fixed << std::setprecision(3) << x[i] << " ";
        std::cout << "\n";

        std::cout << "Время выполнения = " << end_time - start_time << " с.\n"; // Вывод времени
    }

    MPI_Finalize(); // Завершение работы MPI
    return 0;       // Завершение программы
}


Overwriting practice92.cpp


In [86]:
# Компиляция
!mpic++ practice92.cpp -o practice92
# mpirun — утилита, которая запускает MPI-программу на указанном числе процессов
# --allow-run-as-root — разрешает запуск MPI от root (Colab запускает ядра как root)
# --oversubscribe — игнорирует количество доступных виртуальных CPU, позволяя запускать больше процессов, чем физически есть
# -np 2 — число процессов MPI (2 процесса)

# Запуск с 2 процессами
!mpirun --allow-run-as-root --oversubscribe -np 2 ./practice92 10


Исходная матрица A:
    7     8     6     4     6     7     3    10     2     3 
    1    10     4     7     1     7     3     7     2     9 
   10     3     1     3     4     8     6    10     3     3 
   10     8     4     7     2     3    10     4     2    10 
    8     9     5     6     1     4     7     2     1     7 
    3     1     7     2     6     6     5     8     7     6 
   10     4     8     5     6     3     6     5     8     5 
    4     1     8     9     7     9     9     5     4     2 
   10     3     1     7     9    10     3     7     7     5 
    6     1     5     9     8     2     8     3     8     3 
Исходный вектор b:
    4     8     8     9     5     4     7     5     5    10 
Решение x:
-0.010 0.111 -0.430 0.909 -0.879 -0.537 0.429 0.899 0.835 -0.262 
Время выполнения = 0.000 с.


In [87]:
# Запуск с 8 процессами
!mpirun --allow-run-as-root --oversubscribe -np 8 ./practice92 10


Исходная матрица A:
    7     8     6     4     6     7     3    10     2     3 
    1    10     4     7     1     7     3     7     2     9 
   10     3     1     3     4     8     6    10     3     3 
   10     8     4     7     2     3    10     4     2    10 
    8     9     5     6     1     4     7     2     1     7 
    3     1     7     2     6     6     5     8     7     6 
   10     4     8     5     6     3     6     5     8     5 
    4     1     8     9     7     9     9     5     4     2 
   10     3     1     7     9    10     3     7     7     5 
    6     1     5     9     8     2     8     3     8     3 
Исходный вектор b:
    4     8     8     9     5     4     7     5     5    10 
Решение x:
-0.010 0.111 -0.430 0.909 -0.879 -0.537 0.429 0.899 0.835 -0.262 
Время выполнения = 0.004 с.


In [88]:
# Запуск с 16 процессами
!mpirun --allow-run-as-root --oversubscribe -np 16 ./practice92 10

Исходная матрица A:
    7     8     6     4     6     7     3    10     2     3 
    1    10     4     7     1     7     3     7     2     9 
   10     3     1     3     4     8     6    10     3     3 
   10     8     4     7     2     3    10     4     2    10 
    8     9     5     6     1     4     7     2     1     7 
    3     1     7     2     6     6     5     8     7     6 
   10     4     8     5     6     3     6     5     8     5 
    4     1     8     9     7     9     9     5     4     2 
   10     3     1     7     9    10     3     7     7     5 
    6     1     5     9     8     2     8     3     8     3 
Исходный вектор b:
    4     8     8     9     5     4     7     5     5    10 
Решение x:
-0.010 0.111 -0.430 0.909 -0.879 -0.537 0.429 0.899 0.835 -0.262 
Время выполнения = 0.006 с.
